In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
def speaker_ids(names):
    d_names = []
    re_names = []
    for id, re_n in enumerate(names):
        n = f"SPK{id}"
        re_names.append(re_n)
        d_names.append(n)
    return dict(zip(re_names,d_names))


In [3]:
candor_ds_path = "/mas/projects/prg-candor/"
dialogues = []
dialogues_ids = []
maps = []

for root, dirs, files in os.walk(candor_ds_path):
    for file in files:
        if file.endswith('transcript_audiophile.csv'):
            dialogue_id = root.split("/")[-2]
            d = pd.read_csv(os.path.join(root,file))
            utterances = d["utterance"]
            speakers = np.unique(np.asarray(d["speaker"]))
            map_names = speaker_ids(speakers)
            dialogues_ids.append(dialogue_id)
            maps.append(map_names)
            dialogues.append(" ".join(d.apply(lambda x: f"<{map_names[x['speaker']]}> {x['utterance']}", axis=1)))

In [4]:
len(dialogues_ids)

1656

In [5]:
pd.DataFrame({"file_name":dialogues_ids, "file_content":dialogues}).to_csv("/u/sebono/conversational_dominance/data/processed/CANDOR/conversations.csv", index=False)

In [6]:
import json
with open("/u/sebono/conversational_dominance/data/processed/CANDOR/maps.json", "w") as json_file:
    json.dump(maps, json_file, indent=4)

In [6]:
num = len(dialogues_ids) // 8

n_1 = 0
for n in range(1,9):
    pd.DataFrame({"conversation_id": dialogues_ids[n_1*num:n*num]}).to_csv(f"/u/sebono/conversational_dominance/data/processed/CANDOR/group_{n}.csv", index=False)
    n_1+=1